### 01. 试一下 lower func

In [ ]:
import tilelang

def my_func():
    pass

try:
    compiled_artifact = tilelang.lower(my_func)
    print(compiled_artifact.kernel_source)  
except Exception as e:
    print(f"Error: {e}")

### 02. 使用 T.prim_func 装饰的func

In [ ]:
import tilelang
import tilelang.language as T

@T.prim_func
def main():
    pass

compiled_artifact = tilelang.lower(main)
print(compiled_artifact.kernel_source)

### 03. func 改造为 add

In [ ]:
import tilelang
import tilelang.language as T

@T.prim_func
def main(
    A: T.Tensor((1,1), "float32"), # type: ignore
    B: T.Tensor((1,1), "float32"), # type: ignore
    C: T.Tensor((1,1), "float32"), # type: ignore
):
    a_ub = T.alloc_ub((1,1), "float32")
    b_ub = T.alloc_ub((1,1), "float32")
    c_ub = T.alloc_ub((1,1), "float32")

    T.copy(A[0, 0], a_ub)
    T.copy(B[0, 0], b_ub)

    T.tile.add(c_ub, a_ub, b_ub)

    T.copy(c_ub, C[0, 0])

compiled_artifact = tilelang.lower(main)
print(compiled_artifact.kernel_source)

### 04. 配置自动插同步的pass

In [ ]:
import tilelang
import tilelang.language as T
from tilelang import tvm as tvm

pass_configs = {
    tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
}

@T.prim_func
def main(
    A: T.Tensor((1,1), "float32"), # type: ignore
    B: T.Tensor((1,1), "float32"), # type: ignore
    C: T.Tensor((1,1), "float32"), # type: ignore
):
    a_ub = T.alloc_ub((1,1), "float32")
    b_ub = T.alloc_ub((1,1), "float32")
    c_ub = T.alloc_ub((1,1), "float32")

    T.copy(A[0, 0], a_ub)
    T.copy(B[0, 0], b_ub)

    T.tile.add(c_ub, a_ub, b_ub)

    T.copy(c_ub, C[0, 0])

with tvm.transform.PassContext(opt_level=3, config=pass_configs):
    compiled_artifact = tilelang.lower(main)
    print(compiled_artifact.kernel_source)

### 05. 使用 with T.Kernel()

In [ ]:
import tilelang
import tilelang.language as T
from tilelang import tvm as tvm

pass_configs = {
    tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
}

@T.prim_func
def main(
    A: T.Tensor((1,1), "float32"), # type: ignore
    B: T.Tensor((1,1), "float32"), # type: ignore
    C: T.Tensor((1,1), "float32"), # type: ignore
):
    with T.Kernel(1, is_npu=True) as (cid, vid):
        a_ub = T.alloc_ub((1,1), "float32")
        b_ub = T.alloc_ub((1,1), "float32")
        c_ub = T.alloc_ub((1,1), "float32")

        T.copy(A[0, 0], a_ub)
        T.copy(B[0, 0], b_ub)

        T.tile.add(c_ub, a_ub, b_ub)

        T.copy(c_ub, C[0, 0])

with tvm.transform.PassContext(opt_level=3, config=pass_configs):
    compiled_artifact = tilelang.lower(main)
    print(compiled_artifact.kernel_source)